# Chain-of-Thought 복습과제 — 중간 추론 단계가 왜 도움이 될까?

**논문:** Wei et al., *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models* (NeurIPS 2022)

이 과제에서는 PaLM 540B나 GPT-3 175B를 재현하지 않습니다. 대신 작은 공개 causal language model과 간단한 문자열 처리로 **standard prompting과 chain-of-thought prompting의 차이**를 확인하고, 논문의 주요 결과와 ablation 설계를 복습합니다.

| Part | 내용 | 논문 연결 |
|---|---|---|
| 1 | Standard / Chain-of-Thought prompt 구성 | §2, Figure 1 |
| 2 | 답 추출 + 작은 LM으로 CoT 관찰 | §3.1, §3.2 |
| 3 | 모델 규모와 문제 난이도에 따른 결과 해석 | §3.2, Table 2–3 |
| 4 | Ablation prompt 직접 만들기 | §3.3, Figure 5, Table 6 |
| 5 | Symbolic reasoning과 길이 일반화 | §5, Table 5 |
| 6 | 서술형 복습 | §6–8 |

> 목표는 코드를 어렵게 구현하는 것이 아니라, **prompt에서 정확히 무엇이 바뀌었고, 어떤 가설을 배제하려는 실험인지** 이해하는 것입니다.

In [ ]:
# 0. 환경 설정
!pip -q install transformers

import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

---
## Part 1. Standard prompt / Chain-of-Thought prompt 만들기

논문에서 두 방식의 차이는 **few-shot exemplar의 답 부분에 chain of thought(중간 추론 단계)를 포함하느냐**입니다.

- Standard prompting: `<input, output>` 쌍
- Chain-of-thought prompting: `<input, chain of thought, output>` triple

두 경우 모두 model weight는 업데이트하지 않습니다. 아래 exemplar는 논문 Appendix Table 20의 수학 문장제 prompt에서 가져왔습니다.

### 문제 1-1. `build_prompt`의 빈칸을 채우세요

- `cot=False`: `A: The answer is {answer}.`
- `cot=True`: `A: {chain} The answer is {answer}.`

In [1]:
exemplars = [
    {
        "question": "There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?",
        "chain": "There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6.",
        "equation": "21 - 15 = 6",
        "answer": "6",
    },
    {
        "question": "If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?",
        "chain": "There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5.",
        "equation": "3 + 2 = 5",
        "answer": "5",
    },
    {
        "question": "Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?",
        "chain": "Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39.",
        "equation": "32 + 42 - 35 = 39",
        "answer": "39",
    },
    {
        "question": "Olivia has $23. She bought five bagels for $3 each. How much money does she have left?",
        "chain": "Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 dollars left. 23 - 15 is 8.",
        "equation": "23 - 5 * 3 = 8",
        "answer": "8",
    },
]

def format_answer(ex, cot):
    if cot:
        return f"{ex['chain']} The answer is {ex['answer']}."                          # TODO ① chain of thought + 최종 답
    return f"The answer is {ex['answer']}."

def build_prompt(exemplars, query, cot=False):
    prompt = ""
    for ex in exemplars:
        prompt += f"Q: {ex['question']}\nA: {format_answer(ex, cot)}\n\n"
    prompt += f"Q: {query}\nA:"
    return prompt

query = "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"

print("===== Standard =====")
print(build_prompt(exemplars[:2], query, cot=False))
print("\n===== Chain-of-Thought =====")
print(build_prompt(exemplars[:2], query, cot=True))

===== Standard =====
Q: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
A: The answer is 6.

Q: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
A: The answer is 5.

Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
A:

===== Chain-of-Thought =====
Q: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
A: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.

Q: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
A: There are originally 3 cars. 2 more cars arrive. 3

In [2]:
# ✅ 확인 셀
std = build_prompt(exemplars, query, cot=False)
cot = build_prompt(exemplars, query, cot=True)
assert "21 - 15 = 6" not in std
assert "21 - 15 = 6. The answer is 6." in cot
assert std.endswith(f"Q: {query}\nA:") and cot.endswith(f"Q: {query}\nA:")
print("✅ prompt 구성 확인")

✅ prompt 구성 확인


### 문제 1-2 (서술형)
Standard prompting → Chain-of-thought prompting으로 바꿀 때 **변하는 것**과 **변하지 않는 것**을 각각 쓰세요. exemplar 개수, test query의 형태, model parameter의 update 여부를 포함해 설명하세요.

**답:** CoT에서는 각 예시의 답 앞에 중간 추론 과정이 추가된다. 예시 개수와 test query는 그대로이고, 두 방식 모두 추론 시 프롬프트만 바꾸기 때문에 모델 파라미터는 업데이트하지 않는다.

---
## Part 2. 답 추출과 작은 Language Model로 CoT 관찰하기

CoT prompting에서는 모델이 긴 추론 문장을 생성하므로, 채점하려면 생성 결과에서 **최종 답만 추출**해야 합니다. 논문의 exemplar는 모두 `The answer is X.`로 끝나므로, 모델도 같은 형식을 따르도록 유도됩니다.

### 문제 2-1. 생성 결과에서 최종 답을 추출하세요

`"The answer is"` 뒤에 나오는 첫 번째 정수를 문자열로 반환하고, 없으면 `None`을 반환합니다.

In [3]:
def extract_answer(text):
    match = re.search(r"The answer is\s+(-?\d+)", text)            # TODO ② "The answer is" 뒤의 (음수 포함) 정수를 찾는 정규식
    return match.group(1) if match else None

print(extract_answer("Roger started with 5 balls. 2 cans of 3 balls is 6. 5 + 6 = 11. The answer is 11."))
print(extract_answer("I am not sure."))

11
None


In [4]:
# ✅ 확인 셀
assert extract_answer("5 + 6 = 11. The answer is 11.") == "11"
assert extract_answer("The answer is -3. Q: next") == "-3"
assert extract_answer("There are 15 trees. 21 - 15 = 6.") is None
print("✅ 답 추출 확인")

✅ 답 추출 확인


### 문제 2-2. 같은 query를 standard / CoT prompt로 비교하세요

실제 논문의 모델은 공개되어 있지 않거나 너무 큽니다. 여기서는 가벼운 `distilgpt2`(약 82M parameter)를 사용해 **prompt만 바꿨을 때 출력이 어떻게 달라지는지** 관찰합니다.

> 작은 모델이 CoT로 문제를 풀지 못하는 것은 정상입니다. 오히려 이것이 논문의 중요한 관찰과 연결됩니다.

In [ ]:
MODEL_ID = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device).eval()
tokenizer.pad_token = tokenizer.eos_token

@torch.no_grad()
def generate(prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,                         # 논문과 같이 greedy decoding
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    completion = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return completion.split("\nQ:")[0].strip()   # 다음 문제를 생성하기 시작하면 자르기

print("model loaded:", MODEL_ID)

In [ ]:
test_set = [
    (query, "11"),
    ("A baker made 12 cookies and ate 4 of them. Then she made 6 more. How many cookies does she have now?", "14"),
]

observations = []
for q, gold in test_set:
    for use_cot in [False, True]:
        prompt = build_prompt(exemplars, q, cot=use_cot)                        # TODO ③ build_prompt로 4-shot prompt 생성 (use_cot 사용)
        completion = generate(prompt)
        pred = extract_answer(completion)
        observations.append((q, gold, use_cot, pred, completion))
        tag = "CoT     " if use_cot else "Standard"
        print(f"[{tag}] pred={pred} gold={gold} | {completion[:120]}")
    print()
# 전체 생성 결과와 정답 일치 여부 확인
print("\n--- 실제 생성 결과 (문제 2-3 참고) ---")
for q, gold, use_cot, pred, completion in observations:
    print(f"\n{'CoT' if use_cot else 'Standard'} | 정답: {gold} | 추출: {pred} | 일치: {pred == gold}")
    print(completion)


### 문제 2-3 (서술형)
1. 위 실험에서 `distilgpt2`가 생성한 chain of thought는 어떤 특징을 보이나요? (형식은 따라 하는지, 계산은 맞는지)
2. 이 결과만 보고 “CoT prompting은 효과가 없다”고 결론 내릴 수 없는 이유를 논문의 Table 2(예: LaMDA 420M, GPT 350M의 결과)와 연결해 쓰세요.

**답:** 1. 위 셀을 실행하면 각 방식의 전체 생성 문장과 정답 일치 여부를 확인할 수 있다. 작은 모델은 예시의 문장 형식을 따라 하더라도 계산이 틀리거나 최종 답 형식까지 생성하지 못할 수 있다. 실제 형식과 계산의 정확성은 위의 출력으로 판단해야 한다.

2. distilgpt2는 약 82M 규모로 논문의 대형 모델과 차이가 크다. Table 2에서도 GSM8K에서 LaMDA 420M은 2.6% → 0.4%, GPT 350M은 2.2% → 0.5%로 CoT를 적용했을 때 오히려 성능이 낮아졌다. 따라서 작은 모델에서 효과가 없더라도 충분히 큰 모델에서 CoT가 유효하다는 논문의 결과를 반박할 수는 없다.

---
## Part 3. 논문의 주요 결과 읽기

아래는 논문 Table 2(GSM8K)와 Table 3(MAWPS subset)의 일부입니다. 숫자를 직접 계산하며 두 가지 주장을 확인합니다.

- **주장 1:** CoT는 모델 규모에 따른 *emergent ability*이다.
- **주장 2:** 문제가 복잡할수록 CoT의 이득이 크다.

### 문제 3-1. 모델 크기별 CoT 이득을 계산하세요

In [5]:
# GSM8K accuracy (%) — Table 2
gsm8k = {
    "LaMDA": {"420M": (2.6, 0.4), "2B": (3.6, 1.9), "8B": (3.2, 1.6), "68B": (5.7, 8.2), "137B": (6.5, 14.3)},
    "GPT":   {"350M": (2.2, 0.5), "1.3B": (2.4, 0.5), "6.7B": (4.0, 2.4), "175B": (15.6, 46.9)},
    "PaLM":  {"8B": (4.9, 4.1), "62B": (9.6, 29.9), "540B": (17.9, 56.9)},
}   # (standard, CoT)

def cot_gain(standard, cot):
    return cot - standard                              # TODO ④ CoT - standard

for family, sizes in gsm8k.items():
    row = []
    for size, (s, c) in sizes.items():
        row.append(f"{size}: {cot_gain(s, c):+.1f}")
    print(f"{family:6s} | " + "  ".join(row))

LaMDA  | 420M: -2.2  2B: -1.7  8B: -1.6  68B: +2.5  137B: +7.8
GPT    | 350M: -1.7  1.3B: -1.9  6.7B: -1.6  175B: +31.3
PaLM   | 8B: -0.8  62B: +20.3  540B: +39.0


In [6]:
# ✅ 확인 셀
assert round(cot_gain(*gsm8k["PaLM"]["540B"]), 1) == 39.0
assert round(cot_gain(*gsm8k["GPT"]["350M"]), 1) == -1.7
print("✅ 계산 확인")

✅ 계산 확인


### 문제 3-2. 문제 난이도에 따른 이득을 비교하세요

MAWPS의 SingleOp은 한 단계 연산 문제, MultiArith는 여러 단계 연산이 필요한 문제입니다.

In [7]:
# PaLM 540B, MAWPS subsets — Table 3  (standard, CoT)
mawps_palm540 = {
    "SingleOp":   (94.1, 94.1),
    "SingleEq":   (86.5, 92.3),
    "AddSub":     (93.9, 91.9),
    "MultiArith": (42.2, 94.7),
}

for subset, (s, c) in mawps_palm540.items():
    print(f"{subset:10s} | standard {s:5.1f} → CoT {c:5.1f} | gain {cot_gain(s, c):+.1f}")

SingleOp   | standard  94.1 → CoT  94.1 | gain +0.0
SingleEq   | standard  86.5 → CoT  92.3 | gain +5.8
AddSub     | standard  93.9 → CoT  91.9 | gain -2.0
MultiArith | standard  42.2 → CoT  94.7 | gain +52.5


### 문제 3-3 (서술형)
1. 세 모델 family에서 CoT 이득이 **양수로 바뀌는 지점**은 대략 어느 규모인가요? 작은 모델에서는 왜 오히려 성능이 떨어질 수 있을까요? (논문 §3.2의 설명을 참고하세요.)
2. MAWPS 결과에서 SingleOp과 MultiArith의 이득 차이는 무엇을 보여주나요?

**답:** 1. GSM8K 기준 LaMDA는 68B, GPT는 175B, PaLM은 62B에서 처음 양의 이득이 관찰된다. 작은 모델은 자연스러운 문장을 생성하더라도 논리적으로 연결된 추론이나 계산을 정확히 수행하지 못해 오류가 누적될 수 있다. 이 수치는 실험한 모델에서의 관찰이지 모든 모델의 절대적 임계값은 아니다.

2. SingleOp은 94.1%에서 그대로이고 MultiArith는 42.2%에서 94.7%로 52.5%p 상승했다. 한 단계로 풀 수 있는 문제보다 여러 단계의 계산이 필요한 문제에서 중간 추론을 명시하는 이점이 크다는 것을 보여준다.

---
## Part 4. Ablation prompt 직접 만들기

CoT가 잘 되는 이유에 대해 다른 설명도 가능합니다. 논문은 아래 세 가지 변형 prompt로 각각의 대안 가설을 검사했습니다 (§3.3, Figure 5).

| Ablation | exemplar의 답 형태 | 검사하려는 가설 |
|---|---|---|
| Equation only | `A: 21 - 15 = 6. The answer is 6.` | 수식만 있으면 충분한가? |
| Variable compute only | `A: ........... The answer is 6.` | 토큰(연산량)이 늘어난 덕분인가? |
| Reasoning after answer | `A: The answer is 6. There are 15 trees ...` | 답을 낸 뒤의 설명도 같은 효과인가? |

### 문제 4-1. 세 가지 ablation 형식을 구현하세요

- Variable compute only: 논문은 **수식에 필요한 문자 수만큼의 점(`.`)**을 사용했습니다. 여기서는 `equation` 문자열 길이만큼 점을 찍습니다.

In [8]:
def format_ablation(ex, mode):
    if mode == "equation_only":
        return f"{ex['equation']}. The answer is {ex['answer']}."
    if mode == "variable_compute":
        dots = "." * len(ex["equation"])                          # TODO ⑤ equation 길이만큼의 "."
        return f"{dots} The answer is {ex['answer']}."
    if mode == "reasoning_after_answer":
        return f"The answer is {ex['answer']}. {ex['chain']}"                          # TODO ⑥ 답을 먼저, chain of thought를 뒤에
    raise ValueError(mode)

def build_ablation_prompt(exemplars, query, mode):
    prompt = ""
    for ex in exemplars:
        prompt += f"Q: {ex['question']}\nA: {format_ablation(ex, mode)}\n\n"
    prompt += f"Q: {query}\nA:"
    return prompt

for mode in ["equation_only", "variable_compute", "reasoning_after_answer"]:
    print(f"--- {mode} ---")
    print(format_ablation(exemplars[0], mode), "\n")

--- equation_only ---
21 - 15 = 6. The answer is 6. 

--- variable_compute ---
........... The answer is 6. 

--- reasoning_after_answer ---
The answer is 6. There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. 



In [9]:
# ✅ 확인 셀
ex0 = exemplars[0]
assert format_ablation(ex0, "variable_compute") == "." * len(ex0["equation"]) + " The answer is 6."
after = format_ablation(ex0, "reasoning_after_answer")
assert after.startswith("The answer is 6.") and ex0["chain"] in after
assert after.index("The answer is") < after.index(ex0["chain"])
print("✅ ablation prompt 확인")

✅ ablation prompt 확인


### 문제 4-2. 논문의 ablation 결과를 정리하세요

In [10]:
# LaMDA 137B, GSM8K accuracy (%) — Table 6
ablation_gsm8k = {
    "standard": 6.5,
    "equation_only": 5.4,
    "variable_compute": 6.4,
    "reasoning_after_answer": 6.1,
    "chain_of_thought": 14.3,
}

best_ablation = max((k for k in ablation_gsm8k if k not in ("standard", "chain_of_thought")), key=ablation_gsm8k.get)                         # TODO ⑦ standard, chain_of_thought를 제외한 ablation 중 가장 높은 것의 이름
print("best ablation:", best_ablation, ablation_gsm8k[best_ablation])
print("CoT - best ablation:", round(ablation_gsm8k["chain_of_thought"] - ablation_gsm8k[best_ablation], 1))

best ablation: variable_compute 6.4
CoT - best ablation: 7.9


In [11]:
# ✅ 확인 셀
assert best_ablation == "variable_compute"
print("✅ 결과 확인")

✅ 결과 확인


### 문제 4-3 (서술형)
1. 세 ablation은 각각 어떤 대안 가설을 **배제**하나요? 위 결과를 근거로 한 문장씩 쓰세요.
2. Equation only는 GSM8K에서는 거의 도움이 되지 않았지만, 한두 단계로 풀리는 쉬운 데이터셋(SVAMP, ASDiv 등)에서는 성능을 올렸습니다 (Table 6). 이 차이는 무엇을 의미할까요?

**답:** 1. Equation only(5.4%)는 수식만으로 CoT의 이득을 설명하기 어렵다는 것을 보여준다. Variable compute only(6.4%)는 의미 없는 점을 추가하는 방식의 연산량 증가만으로는 충분하지 않음을 보여준다. Reasoning after answer(6.1%)는 답을 먼저 생성한 뒤 설명하는 것만으로는 CoT의 효과를 얻기 어렵다는 것을 보여준다. 세 조건 모두 standard(6.5%)와 비슷하거나 낮고 CoT(14.3%)보다 낮지만, 모든 가능한 대안 가설이 완전히 배제된 것은 아니다.

2. 간단한 문제는 입력에서 바로 수식을 구성할 수 있어 수식만으로도 도움이 된다. GSM8K처럼 복잡한 문장제는 문제의 의미와 조건을 단계별로 이해해야 올바른 수식을 세울 수 있어 자연어 추론 과정의 역할이 더 크다.

---
## Part 5. Symbolic Reasoning과 길이 일반화

Last letter concatenation은 이름에 포함된 각 단어의 마지막 글자를 이어 붙이는 task입니다.
예: `"Elon Musk"` → `"nk"`

논문은 exemplar를 **2단어 이름**으로만 구성하고, test에서 **3·4단어 이름(OOD)**을 사용해 길이 일반화를 확인했습니다.

### 문제 5-1. 정답 함수와 chain of thought 생성 함수를 완성하세요

`make_chain`은 논문 Table 22의 exemplar 형식을 그대로 따릅니다.

In [12]:
def last_letter_concat(name):
    return "".join(word[-1] for word in name.split())                              # TODO ⑧ 각 단어의 마지막 글자를 이어 붙이기

def make_chain(name):
    words = name.split()
    steps = " ".join(f'The last letter of "{w}" is "{w[-1]}".' for w in words)
    ans = last_letter_concat(name)
    return f'{steps} Concatenating them is "{ans}". The answer is {ans}.'

def letter_prompt(name):
    return f'Q: Take the last letters of the words in "{name}" and concatenate them.\nA:'

print(make_chain("Elon Musk"))
print(make_chain("Bill Henry Gates"))   # OOD: 3단어

The last letter of "Elon" is "n". The last letter of "Musk" is "k". Concatenating them is "nk". The answer is nk.
The last letter of "Bill" is "l". The last letter of "Henry" is "y". The last letter of "Gates" is "s". Concatenating them is "lys". The answer is lys.


In [13]:
# ✅ 확인 셀
assert last_letter_concat("Elon Musk") == "nk"
assert last_letter_concat("Larry Page") == "ye"
assert last_letter_concat("Bill Henry Gates") == "lys"
assert make_chain("Sergey Brin").endswith('Concatenating them is "yn". The answer is yn.')
print("✅ symbolic task 확인")

✅ symbolic task 확인


In [14]:
# Last letter concatenation accuracy (%) — Table 5, PaLM  (standard, CoT)
letter_concat = {
    "8B":   {"in-domain (2)": (2.6, 18.8), "OOD (3)": (0.0, 0.0),  "OOD (4)": (0.0, 0.2)},
    "62B":  {"in-domain (2)": (6.8, 85.0), "OOD (3)": (0.0, 59.6), "OOD (4)": (0.0, 13.4)},
    "540B": {"in-domain (2)": (7.6, 99.4), "OOD (3)": (0.2, 94.8), "OOD (4)": (0.0, 63.0)},
}

for size, res in letter_concat.items():
    print(f"PaLM {size:5s} | " + " | ".join(f"{k}: {s:4.1f} → {c:4.1f}" for k, (s, c) in res.items()))

PaLM 8B    | in-domain (2):  2.6 → 18.8 | OOD (3):  0.0 →  0.0 | OOD (4):  0.0 →  0.2
PaLM 62B   | in-domain (2):  6.8 → 85.0 | OOD (3):  0.0 → 59.6 | OOD (4):  0.0 → 13.4
PaLM 540B  | in-domain (2):  7.6 → 99.4 | OOD (3):  0.2 → 94.8 | OOD (4):  0.0 → 63.0


### 문제 5-2 (서술형)
1. Standard prompting은 OOD에서 거의 0%인데 CoT는 PaLM 540B에서 94.8%(3단어)를 기록합니다. CoT의 어떤 구조가 **exemplar보다 긴 입력**으로의 일반화를 가능하게 할까요? 위의 `make_chain` 출력 형태를 참고하세요.
2. 이 task는 정답을 구하는 절차가 exemplar에 그대로 드러나 있는 “toy task”입니다. 그런데도 PaLM 8B는 in-domain에서 18.8%에 그칩니다. 이 결과는 무엇을 시사하나요?

**답:** 1. 각 단어에서 마지막 글자를 추출하고 이를 순서대로 연결하는 반복 가능한 절차가 CoT 예시에 명시되어 있다. 따라서 모델이 이 절차를 새로운 단어에도 반복 적용하면 예시보다 긴 입력을 처리할 수 있다. PaLM 540B에서 3단어 OOD는 94.8%, 4단어는 63.0%로, 길이가 길어져도 어느 정도 일반화되지만 완벽하지는 않다.

2. 절차가 예시에 제시되어도 작은 모델은 그 절차를 새로운 기호에 정확히 적용하지 못할 수 있다. 예시의 형식 모방과 규칙의 정확한 수행은 다르며, 논문의 실험에서는 모델 규모에 따라 이 능력이 달라졌다.

---
## Part 6. 서술형 복습

**6-1.** CoT prompting과 Ling et al. (2017), Cobbe et al. (2021)처럼 rationale을 생성하도록 모델을 학습/파인튜닝하는 방법의 가장 중요한 차이는 무엇인가요? 각 방법의 비용 측면을 포함해 쓰세요.

**답:** 기존 rationale 기반 방법은 풀이가 포함된 데이터를 많이 준비해 모델을 학습하거나 fine-tuning하므로 데이터 작성 및 학습 비용이 든다. CoT prompting은 소수의 풀이 예시를 프롬프트에 넣어 이미 학습된 모델의 추론을 유도하므로 추가 가중치 업데이트가 필요 없다. 다만 추론 시 프롬프트와 출력이 길어지는 비용은 남는다.

**6-2.** 논문은 “chain of thought가 사람의 사고 과정을 모방하지만, 신경망이 실제로 *reasoning*을 하는지는 열린 질문”이라고 말합니다. 정답을 맞힌 CoT 중에도 추론 과정이 틀린 경우가 있다는 점(Appendix D.1)은 CoT의 해석 가능성(interpretability) 주장에 어떤 한계를 주나요?

**답:** 생성된 CoT는 어디에서 오류가 생겼는지 살펴볼 단서를 주지만, 모델 내부의 실제 계산 과정을 그대로 드러낸다고 볼 수는 없다. LaMDA 137B의 GSM8K 정답 샘플 50개 중 2개는 풀이가 틀렸는데 우연히 정답에 도달했다. 따라서 최종 답이 맞거나 설명이 그럴듯하다는 사실만으로 추론 과정의 정확성과 충실성을 보장할 수 없다.

**6-3.** 논문은 “standard prompting은 대형 언어 모델 능력의 *lower bound*일 뿐”이라고 말합니다. 이 주장의 의미를 Part 3의 결과와 연결해 설명하세요.

**답:** 같은 모델이라도 프롬프트 구성에 따라 드러나는 능력이 달라질 수 있다는 의미다. PaLM 540B의 GSM8K 정확도가 standard 17.9%에서 CoT 56.9%로 증가한 것처럼, standard prompting 결과만으로 모델이 수행할 수 있는 추론의 수준을 모두 판단하기는 어렵다.

**6-4.** CoT prompting은 정확도를 높이지만, 모델이 생성하는 토큰 수를 크게 늘립니다. 실제 서비스 관점에서 이것이 어떤 비용(latency, 메모리 등)을 만드는지 쓰고, 논문의 Limitations에서 언급한 “큰 모델에서만 효과가 나타난다”는 점과 함께 실용성을 평가하세요.

**답:** CoT는 출력 토큰이 늘어 autoregressive decoding 시간과 비용이 증가하고, 긴 문맥을 유지하기 위한 KV cache 메모리도 커질 수 있다. 논문에서 큰 모델일수록 효과가 뚜렷했다는 점까지 고려하면 서비스 비용이 부담될 수 있다. 쉬운 문제는 짧게 답하도록 하고 복잡한 문제에만 CoT를 사용하는 등 정확도와 비용을 함께 고려해야 한다.

---
## 참고 자료
- Wei et al. (2022), *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models*: CoT Prompting (§2), Arithmetic Reasoning (§3), Commonsense Reasoning (§4), Symbolic Reasoning (§5), Discussion (§6)
- Appendix B (Table 1–7): 모델·규모별 전체 결과, Appendix G: 실험에 사용한 전체 few-shot prompt

> 이 과제는 대형 모델의 원 실험을 재현하는 과제가 아니라, CoT prompting의 핵심 아이디어와 실험 설계를 작은 예제로 복습하기 위한 과제입니다.